In [7]:
import os
import tushare as ts
pro = ts.pro_api(os.environ['TUSHARE_TOKEN'])    # Mytoken, keep it safe

In [3]:
df = pro.daily(ts_code='002230.SZ', start_date='20260301', end_date='20260421')
df

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
0,002230.SZ,20260421,49.50,49.64,48.40,48.78,48.94,-0.16,-0.3269,327475.64,1597300.299
1,002230.SZ,20260420,48.20,49.07,47.90,48.94,48.24,0.70,1.4511,434959.77,2120636.423
2,002230.SZ,20260417,47.86,48.45,47.64,48.24,48.10,0.14,0.2911,290996.50,1399841.760
3,002230.SZ,20260416,47.83,48.46,47.80,48.10,47.54,0.56,1.1780,351349.60,1689317.381
4,002230.SZ,20260415,48.86,48.98,47.39,47.54,48.33,-0.79,-1.6346,359789.85,1729037.367
5,002230.SZ,20260414,48.50,48.64,47.87,48.33,47.78,0.55,1.1511,366268.03,1768145.918
6,002230.SZ,20260413,47.23,47.92,47.06,47.78,47.64,0.14,0.2939,278604.75,1326252.523
7,002230.SZ,20260410,47.50,48.33,47.47,47.64,47.21,0.43,0.9108,390025.42,1866549.559
8,002230.SZ,20260409,47.26,47.41,46.80,47.21,47.91,-0.70,-1.4611,302452.50,1426238.984
9,002230.SZ,20260408,46.50,47.92,46.50,47.91,45.58,2.33,5.1119,524068.41,2482663.823


In [ ]:
# # save index_basic
# df = pro.index_basic(market='SW')
# df.to_csv('index_basic_ts_20260519.csv', index=False, encoding='utf-8-sig')
# # save data and draw
# C:\Users\junyu\OneDrive\桌面\Quant\Astock\sw1_kline.py

# 从个人认知和趋势选一级市场: target未来5年-直到下一个十五五规划出来
# 换手率和量比：计算机、电力设备、社会服务、公用事业、机械设备、国防军工、汽车、基础化工、有色金属
# C:\Users\junyu\OneDrive\桌面\Quant\Astock\sw1_turnover_vol.py

#### 看看二级市场
# C:\Users\junyu\OneDrive\桌面\Quant\Astock\sw2_kline.py

In [ ]:
# # 行业 + BB + 均线 + 换手率 +量比 + email提醒

# 上升行业: 人为判断? (有没有办法tushare/东方财富获得)

# 过去10天换手率>5% or 前一天换手率>5%
# 量比 不强求
# 日内波动率要够

In [ ]:
# 电力行业调研
"""
申万二级行业"电力"(801161) 全部 A 股成分股
- 成分股列表：akshare index_component_sw（单次调用）
- 日K线下载：tushare pro.daily（更稳定，自带重试）
- 保存 CSV 到 data_Elec/
- 画图：上图 K线+BB+MA5/10/20+资金净流入代理（右轴） / 下图 MACD
"""
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import akshare as ak
import tushare as ts

warnings.filterwarnings("ignore")

TOKEN = os.environ["TUSHARE_TOKEN"]

# ── 配置 ─────────────────────────────────────────────────────────────────────
OUT_DIR    = r"C:\Users\junyu\OneDrive\桌面\Quant\Astock\Astocks\data_Elec"
PLOT_DIR   = os.path.join(OUT_DIR, "plots")
START_DATE = "20160101"
END_DATE   = pd.Timestamp.today().strftime("%Y%m%d")
os.makedirs(OUT_DIR,  exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family":        "Microsoft YaHei",
    "axes.unicode_minus": False,
    "figure.dpi":         150,
})

pro = ts.pro_api(TOKEN)


# ── 1. 获取电力行业成分股（申万二级 801161）────────────────────────────────
print("获取电力行业成分股 (801161)...")
df_comp = ak.index_component_sw(symbol="801161")
df_comp.columns = ["rank", "code", "name", "weight", "in_date"]
df_comp["code"] = df_comp["code"].astype(str).str.zfill(6)
# 判断市场后缀：60/68 开头→.SH，否则→.SZ
def ts_code(c):
    return c + (".SH" if c[:2] in ("60", "68", "90") else ".SZ")
df_comp["ts_code"] = df_comp["code"].apply(ts_code)
df_comp.to_csv(r"C:\Users\junyu\OneDrive\桌面\Quant\Astock\Astocks\Elec\stocks.csv", index=False, encoding="utf-8-sig")

获取电力行业成分股 (801161)...


In [8]:
# -*- coding: utf-8 -*-
"""
电力行业股票 —— 方向1 & 2 分析
  散点图：pct_chg vs turnover_rate / 量比
  四象限：放量涨/跌、缩量涨/跌 → 次日收益分布
  滞后相关性：corr(indicator[t], pct_chg[t+k])  k=0..5
"""
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as mgrid

def linregress_np(x, y):
    """返回 (slope, intercept, r, p_approx)，纯 numpy 实现。"""
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    n = len(x)
    if n < 3:
        return 0.0, 0.0, 0.0, 1.0
    mx, my = x.mean(), y.mean()
    ss_xy  = ((x - mx) * (y - my)).sum()
    ss_xx  = ((x - mx) ** 2).sum()
    ss_yy  = ((y - my) ** 2).sum()
    slope     = ss_xy / ss_xx if ss_xx else 0.0
    intercept = my - slope * mx
    r = ss_xy / np.sqrt(ss_xx * ss_yy) if ss_xx * ss_yy > 0 else 0.0
    t = r * np.sqrt(n - 2) / np.sqrt(max(1 - r**2, 1e-12))
    p = 2 / (1 + abs(t) * np.sqrt(2 / n))   # Gaussian tail 近似
    return float(slope), float(intercept), float(r), float(p)

def pearsonr_np(x, y):
    """Pearson r，纯 numpy。"""
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < 3:
        return 0.0
    r = np.corrcoef(x, y)[0, 1]
    return float(r) if np.isfinite(r) else 0.0

warnings.filterwarnings("ignore")

# ── 路径（__file__ 推导，无中文硬编码）─────────────────────────────────────
SCRIPT_DIR = r"C:\Users\junyu\OneDrive\桌面\Quant\Astock"
DATA_DIR   = os.path.join(SCRIPT_DIR, "Astocks", "Elec", "dataTushare")
OUT_DIR    = os.path.join(SCRIPT_DIR, "Astocks", "Elec", "analysis")
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family":        "Microsoft YaHei",
    "axes.unicode_minus": False,
    "figure.dpi":         150,
})

# ── 1. 加载全部数据 ───────────────────────────────────────────────────────────
print("加载数据...")
frames = []
for fname in sorted(os.listdir(DATA_DIR)):
    if not fname.endswith(".csv"):
        continue
    df = pd.read_csv(os.path.join(DATA_DIR, fname))
    df["trade_date"] = pd.to_datetime(df["trade_date"].astype(str))
    df = df.sort_values("trade_date").reset_index(drop=True)
    # 量比 = 当日量 / 5日滚动均量（shift 1 避免用当日数据）
    df["vol_ma5"]    = df["vol"].rolling(5, min_periods=3).mean().shift(1)
    df["vol_ratio"]  = df["vol"] / df["vol_ma5"]
    for col in ["pct_chg", "turnover_rate", "vol_ratio"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    frames.append(df)

all_df = pd.concat(frames, ignore_index=True)
print(f"全量数据: {len(all_df):,} 行，{all_df['ts_code'].nunique()} 只股票")

# ── 近一个月（最近 22 个交易日）─────────────────────────────────────────────
recent_dates = sorted(all_df["trade_date"].unique())[-22:]
month_df = all_df[all_df["trade_date"].isin(recent_dates)].copy()
month_df = month_df.dropna(subset=["pct_chg", "turnover_rate", "vol_ratio"])
print(f"近一个月数据: {len(month_df):,} 行")


加载数据...
全量数据: 228,451 行，102 只股票
近一个月数据: 2,244 行


In [10]:
print(month_df)

          ts_code trade_date   open   high    low  close  pre_close  change  \
2503    000027.SZ 2026-04-27   6.96   7.02   6.86   6.94       6.93    0.01   
2504    000027.SZ 2026-04-28   6.90   6.97   6.83   6.93       6.94   -0.01   
2505    000027.SZ 2026-04-29   6.88   7.00   6.86   6.96       6.93    0.03   
2506    000027.SZ 2026-04-30   6.93   6.95   6.82   6.84       6.96   -0.12   
2507    000027.SZ 2026-05-06   6.88   7.04   6.80   7.04       6.84    0.20   
...           ...        ...    ...    ...    ...    ...        ...     ...   
228446  605580.SH 2026-05-25  22.10  24.31  22.10  24.31      22.10    2.21   
228447  605580.SH 2026-05-26  24.66  26.70  23.84  25.90      24.31    1.59   
228448  605580.SH 2026-05-27  25.00  26.65  24.61  25.62      25.90   -0.28   
228449  605580.SH 2026-05-28  25.50  28.18  25.00  28.18      25.62    2.56   
228450  605580.SH 2026-05-29  28.01  31.00  26.66  30.29      28.18    2.11   

        pct_chg        vol       amount  turnover_r

In [ ]:
# ── 2. 散点图 + 线性回归 ─────────────────────────────────────────────────────
print("\n绘制散点图...")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("电力行业近一个月 量价散点图", fontsize=14, fontweight="bold")
for ax, xcol, xlabel in [
    (axes[0], "turnover_rate", "换手率 (%)"),
    (axes[1], "vol_ratio",     "量比 (当日量/5日均量)"),
]:
    x = month_df[xcol].clip(
        lower=month_df[xcol].quantile(0.01),
        upper=month_df[xcol].quantile(0.99)
    )
    y = month_df["pct_chg"].clip(-10, 10)

    # 按涨跌着色
    colors = np.where(y >= 0, "#d63031", "#00b894")
    ax.scatter(x, y, c=colors, alpha=0.35, s=14, linewidths=0)

    # 线性回归
    mask = x.notna() & y.notna()
    slope, intercept, r, p = linregress_np(x[mask], y[mask])
    xrange = np.linspace(x[mask].min(), x[mask].max(), 200)
    ax.plot(xrange, slope * xrange + intercept, color="navy", lw=1.8, label=f"r={r:.3f}  p={p:.3f}")

    ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel("当日涨跌幅 (%)", fontsize=10)
    ax.set_title(f"{xlabel}  vs  当日涨跌幅", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)

plt.tight_layout()
p1 = os.path.join(OUT_DIR, "01_scatter_price_vs_indicators.png")
fig.savefig(p1, bbox_inches="tight", dpi=180)
plt.close()
print(f"  saved: {p1}")


# ── 3. 四象限分析（次日收益分布）────────────────────────────────────────────
print("\n计算四象限...")

# 构建 (date, ts_code) → next-day pct_chg
all_sorted = all_df.sort_values(["ts_code", "trade_date"])
all_sorted["next_pct"] = all_sorted.groupby("ts_code")["pct_chg"].shift(-1)

month_q = all_sorted[all_sorted["trade_date"].isin(recent_dates)].dropna(
    subset=["pct_chg", "turnover_rate", "vol_ratio", "next_pct"]
).copy()

# 量比中位数为分界
vr_med = month_q["vol_ratio"].median()
pc_zero = 0.0

def quadrant(row):
    up   = row["pct_chg"] >= pc_zero
    high = row["vol_ratio"] >= vr_med
    if up and high:   return "放量上涨"
    if up and not high: return "缩量上涨"
    if not up and high: return "放量下跌"
    return "缩量下跌"

month_q["quadrant"] = month_q.apply(quadrant, axis=1)
ORDER = ["放量上涨", "缩量上涨", "放量下跌", "缩量下跌"]
COLORS = ["#d63031", "#ff7675", "#00b894", "#55efc4"]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("电力行业近一个月  四象限分析（量比中位数分界）", fontsize=14, fontweight="bold")

# 左图：各象限次日平均收益 + 置信区间
ax = axes[0]
means, cis, counts = [], [], []
for q in ORDER:
    g = month_q[month_q["quadrant"] == q]["next_pct"]
    counts.append(len(g))
    means.append(g.mean())
    se = g.std() / np.sqrt(len(g)) if len(g) > 1 else 0
    cis.append(1.96 * se)

bars = ax.bar(ORDER, means, color=COLORS, alpha=0.85, width=0.6)
ax.errorbar(ORDER, means, yerr=cis, fmt="none", color="black", capsize=5, lw=1.5)
ax.axhline(0, color="gray", lw=0.8, ls="--")
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f"n={cnt}", ha="center", fontsize=8)
ax.set_ylabel("次日平均涨跌幅 (%)", fontsize=10)
ax.set_title("四象限  →  次日平均收益（95% CI）", fontsize=11)
ax.grid(True, axis="y", alpha=0.3)

# 右图：Box plot
ax2 = axes[1]
data_boxes = [month_q[month_q["quadrant"] == q]["next_pct"].clip(-10, 10).values for q in ORDER]
bp = ax2.boxplot(data_boxes, patch_artist=True, notch=False,
                 medianprops=dict(color="white", lw=2),
                 whiskerprops=dict(lw=1.2),
                 flierprops=dict(marker=".", markersize=3, alpha=0.4))
for patch, color in zip(bp["boxes"], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax2.set_xticklabels(ORDER, fontsize=9)
ax2.axhline(0, color="gray", lw=0.8, ls="--")
ax2.set_ylabel("次日涨跌幅 (%)", fontsize=10)
ax2.set_title("四象限  →  次日收益分布（箱形图）", fontsize=11)
ax2.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
p2 = os.path.join(OUT_DIR, "02_quadrant_next_day_return.png")
fig.savefig(p2, bbox_inches="tight", dpi=180)
plt.close()
print(f"  saved: {p2}")

# 打印象限统计
print("\n【四象限统计】")
for q, m, c_i, n in zip(ORDER, means, cis, counts):
    print(f"  {q:8s}  n={n:4d}  次日均收益={m:+.3f}%  ±{c_i:.3f}%")

# ── 4. 滞后相关性 corr(indicator[t], pct_chg[t+k]) ─────────────────────────
print("\n计算滞后相关性（全历史数据）...")
LAGS = [0, 1, 2, 3, 5]
INDICATORS = {
    "换手率 (turnover_rate)": "turnover_rate",
    "量比 (vol_ratio)":      "vol_ratio",
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
fig.suptitle("电力行业  滞后相关性  corr(indicator[t], pct_chg[t+k])\n（逐股计算后取均值±1σ）",
             fontsize=13, fontweight="bold")

for ax, (label, col) in zip(axes, INDICATORS.items()):
    lag_corrs = {k: [] for k in LAGS}

    for fname in sorted(os.listdir(DATA_DIR)):
        if not fname.endswith(".csv"):
            continue
        df = pd.read_csv(os.path.join(DATA_DIR, fname))
        df["trade_date"] = pd.to_datetime(df["trade_date"].astype(str))
        df = df.sort_values("trade_date").reset_index(drop=True)
        df["vol_ma5"]   = df["vol"].rolling(5, min_periods=3).mean().shift(1)
        df["vol_ratio"] = df["vol"] / df["vol_ma5"]
        for c in ["pct_chg", "turnover_rate", "vol_ratio"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        for k in LAGS:
            x = df[col]
            y = df["pct_chg"].shift(-k)   # future return k days later
            valid = x.notna() & y.notna()
            if valid.sum() < 30:
                continue
            r = pearsonr_np(x[valid], y[valid])
            lag_corrs[k].append(r)

    means_lag = [np.mean(lag_corrs[k]) if lag_corrs[k] else np.nan for k in LAGS]
    stds_lag  = [np.std(lag_corrs[k])  if lag_corrs[k] else np.nan for k in LAGS]

    ax.bar(LAGS, means_lag, color=["#0984e3" if m >= 0 else "#d63031" for m in means_lag],
           alpha=0.7, width=0.6, label="均值相关系数")
    ax.errorbar(LAGS, means_lag, yerr=stds_lag, fmt="none",
                color="black", capsize=6, lw=1.5, label="±1σ（跨股票）")
    ax.axhline(0, color="gray", lw=0.8, ls="--")

    for k, m, s in zip(LAGS, means_lag, stds_lag):
        if not np.isnan(m):
            ax.text(k, m + (0.003 if m >= 0 else -0.005),
                    f"{m:+.3f}", ha="center", fontsize=8.5, fontweight="bold")

    ax.set_xticks(LAGS)
    ax.set_xticklabels([f"t+{k}" for k in LAGS], fontsize=10)
    ax.set_xlabel("滞后期 k（交易日）", fontsize=10)
    ax.set_ylabel("Pearson 相关系数", fontsize=10)
    ax.set_title(f"{label}\n→ pct_chg[t+k]", fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, axis="y", alpha=0.3)
    ax.set_ylim(-0.15, 0.15)

plt.tight_layout()
p3 = os.path.join(OUT_DIR, "03_lag_correlation.png")
fig.savefig(p3, bbox_inches="tight", dpi=180)
plt.close()
print(f"  saved: {p3}")

# ── 5. 换手率分位数 → 后续累计收益（分层回测）─────────────────────────────
print("\n计算换手率分层 → 后续累计收益...")

# 对近一个月每日横截面：按换手率分三组（低/中/高），计算次日平均收益
results = []
for date in recent_dates[:-1]:   # 去掉最后一天（无次日数据）
    day_df = all_sorted[all_sorted["trade_date"] == date].dropna(
        subset=["turnover_rate", "next_pct"])
    if len(day_df) < 10:
        continue
    day_df = day_df.copy()
    day_df["tr_tier"] = pd.qcut(day_df["turnover_rate"], 3, labels=["低换手","中换手","高换手"])
    g = day_df.groupby("tr_tier", observed=True)["next_pct"].mean()
    g["date"] = date
    results.append(g)

tier_df = pd.DataFrame(results).set_index("date")

fig, ax = plt.subplots(figsize=(14, 5))
TIER_COLORS = {"低换手": "#00b894", "中换手": "#fdcb6e", "高换手": "#d63031"}
for tier, color in TIER_COLORS.items():
    if tier in tier_df.columns:
        cum = tier_df[tier].cumsum()
        ax.plot(tier_df.index, cum, marker="o", markersize=4,
                lw=1.5, color=color, label=f"{tier} (累计)")
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_title("近一个月  换手率三分位 × 次日收益  累计走势", fontsize=12, fontweight="bold")
ax.set_ylabel("累计次日均收益 (%)", fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
plt.xticks(rotation=25, fontsize=8)
plt.tight_layout()
p4 = os.path.join(OUT_DIR, "04_turnover_tier_cumret.png")
fig.savefig(p4, bbox_inches="tight", dpi=180)
plt.close()
print(f"  saved: {p4}")

print("\n══════════════════════════════════")
print("全部分析完成，图片保存至:")
print(f"  {OUT_DIR}")
for p in [p1, p2, p3, p4]:
    print(f"  {os.path.basename(p)}")


绘制散点图...
  saved: C:\Users\junyu\OneDrive\桌面\Quant\Astock\Astocks\Elec\analysis\01_scatter_price_vs_indicators.png

计算四象限...
  saved: C:\Users\junyu\OneDrive\桌面\Quant\Astock\Astocks\Elec\analysis\02_quadrant_next_day_return.png

【四象限统计】
  放量上涨      n= 727  次日均收益=+0.572%  ±0.288%
  缩量上涨      n= 494  次日均收益=+0.432%  ±0.324%
  放量下跌      n= 344  次日均收益=+0.140%  ±0.334%
  缩量下跌      n= 577  次日均收益=+0.872%  ±0.273%

计算滞后相关性（全历史数据）...
  saved: C:\Users\junyu\OneDrive\桌面\Quant\Astock\Astocks\Elec\analysis\03_lag_correlation.png

计算换手率分层 → 后续累计收益...
  saved: C:\Users\junyu\OneDrive\桌面\Quant\Astock\Astocks\Elec\analysis\04_turnover_tier_cumret.png

══════════════════════════════════
全部分析完成，图片保存至:
  C:\Users\junyu\OneDrive\桌面\Quant\Astock\Astocks\Elec\analysis
  01_scatter_price_vs_indicators.png
  02_quadrant_next_day_return.png
  03_lag_correlation.png
  04_turnover_tier_cumret.png


In [ ]:
SCRIPT_DIR = r"C:\Users\junyu\OneDrive\桌面\Quant\Astock"
import json, os

# turnover price T
import os, pandas as pd

DATA_DIR   = os.path.join(SCRIPT_DIR, 'Astocks', 'Elec', 'dataTushare')
STOCKS_CSV = os.path.join(SCRIPT_DIR, 'Astocks', 'Elec', 'stocks.csv')

# ── 加载全部股票数据 ───────────────────────────────────────────────────────
frames = []
for fname in sorted(os.listdir(DATA_DIR)):
    if not fname.endswith('.csv'):
        continue
    df = pd.read_csv(os.path.join(DATA_DIR, fname),
                     usecols=['ts_code','trade_date','close','pct_chg','turnover_rate'])
    frames.append(df)

all_df = pd.concat(frames, ignore_index=True)
all_df['trade_date']    = pd.to_datetime(all_df['trade_date'].astype(str))
all_df['close']         = pd.to_numeric(all_df['close'],         errors='coerce')
all_df['pct_chg']       = pd.to_numeric(all_df['pct_chg'],       errors='coerce')
all_df['turnover_rate'] = pd.to_numeric(all_df['turnover_rate'], errors='coerce')

# 最近5个交易日
last5  = sorted(all_df['trade_date'].unique())[-5:]
dates  = [str(d.date()) for d in last5]
print("最近5个交易日:", dates)
recent = all_df[all_df['trade_date'].isin(last5)]
# 取每只股票最新收盘价
latest_close = (recent.sort_values('trade_date')
                       .groupby('ts_code')['close']
                       .last()
                       .rename('latest_close')
                       .reset_index())

# ── 条件1：过去5天换手率均值 > 3% ─────────────────────────────────────────
tr_mean      = recent.groupby('ts_code')['turnover_rate'].mean().rename('tr_mean_5d').reset_index()
df_turnover  = tr_mean[tr_mean['tr_mean_5d'] > 3].sort_values('tr_mean_5d', ascending=False)

# ── 条件2：过去5天任意一天涨跌幅 > 5% ────────────────────────────────────
max_pct  = recent.groupby('ts_code')['pct_chg'].max().rename('max_pct_5d').reset_index()
df_p     = max_pct[max_pct['max_pct_5d'] > 5].sort_values('max_pct_5d', ascending=False)

# ── 合并名称 ──────────────────────────────────────────────────────────────
df_meta     = pd.read_csv(STOCKS_CSV, dtype=str)[['ts_code','name','weight']]
df_meta     = df_meta.merge(latest_close, on='ts_code', how='left')
df_turnover = df_turnover.merge(df_meta, on='ts_code', how='left')
df_p        = df_p.merge(df_meta, on='ts_code', how='left')

print(f"\\n换手率均值 >5%（过去5天）：{len(df_turnover)} 只")
print(f"任意一天涨跌幅 >5%（过去5天）：{len(df_p)} 只")

# ── 重合 ──────────────────────────────────────────────────────────────────
overlap_codes = set(df_turnover['ts_code']) & set(df_p['ts_code'])
df_overlap = (df_turnover[df_turnover['ts_code'].isin(overlap_codes)]
              .merge(df_p[['ts_code','max_pct_5d']], on='ts_code')
              .sort_values('tr_mean_5d', ascending=False))

print(f"\\n{'='*50}")
print(f"两者重合：{len(df_overlap)} 只")
print('='*50)
display(df_overlap[['ts_code','name','tr_mean_5d','max_pct_5d','weight','latest_close']].reset_index(drop=True))

print("\\n── 换手率 >3% 全部 ──")
display(df_turnover[['ts_code','name','tr_mean_5d','weight','latest_close']].reset_index(drop=True))

print("\\n── 涨跌幅 >5% 全部 ──")
display(df_p[['ts_code','name','max_pct_5d','weight','latest_close']].reset_index(drop=True))


最近5个交易日: ['2026-05-25', '2026-05-26', '2026-05-27', '2026-05-28', '2026-05-29']
\n换手率均值 >5%（过去5天）：45 只
任意一天涨跌幅 >5%（过去5天）：45 只
\n==================================================
两者重合：29 只


,ts_code,name,tr_mean_5d,max_pct_5d,weight,latest_close
0,600396.SH,华电辽能,20.01454,10.0275,1.1314,20.03
1,600719.SH,大连热电,18.89318,6.8478,0.2235,9.64
2,000601.SZ,韶能股份,15.93526,9.9863,0.5096,8.46
3,600310.SH,广西能源,15.73688,10.0000,0.3275,5.98
4,000767.SZ,晋控电力,13.23784,10.0592,0.501,5.58
5,001896.SZ,豫能控股,12.77014,7.3604,0.8139,16.37
6,001376.SZ,百通能源,11.94186,10.0072,0.2501,14.56
7,300317.SZ,珈伟新能,10.40842,11.2840,0.2757,5.72
8,600744.SH,华银电力,10.32372,9.9744,0.6572,9.06
9,605580.SH,恒盛能源,10.09758,10.0000,0.188,30.29


\n── 换手率 >5% 全部 ──


,ts_code,name,tr_mean_5d,weight,latest_close
0,600396.SH,华电辽能,20.01454,1.1314,20.03
1,600719.SH,大连热电,18.89318,0.2235,9.64
2,002617.SZ,露笑科技,17.78778,1.2121,9.23
3,002893.SZ,京能热力,16.45894,0.1718,12.54
4,000601.SZ,韶能股份,15.93526,0.5096,8.46
5,600310.SH,广西能源,15.73688,0.3275,5.98
6,000767.SZ,晋控电力,13.23784,0.501,5.58
7,001896.SZ,豫能控股,12.77014,0.8139,16.37
8,001376.SZ,百通能源,11.94186,0.2501,14.56
9,300317.SZ,珈伟新能,10.40842,0.2757,5.72


\n── 涨跌幅 >5% 全部 ──


,ts_code,name,max_pct_5d,weight,latest_close
0,300317.SZ,珈伟新能,11.2840,0.2757,5.72
1,000767.SZ,晋控电力,10.0592,0.501,5.58
2,600032.SH,浙江新能,10.0529,0.3995,10.61
3,600011.SH,华能国际,10.0496,1.9245,8.87
4,000037.SZ,深南电A,10.0478,0.1965,11.50
5,002608.SZ,江苏国信,10.0478,0.4179,9.20
6,000539.SZ,粤电力A,10.0450,0.5295,8.88
7,600726.SH,华电能源,10.0358,0.8196,10.13
8,600396.SH,华电辽能,10.0275,1.1314,20.03
9,600863.SH,华能蒙电,10.0186,1.9034,7.30


In [9]:
df_hidden

,ts_code,tr_mean_5d,max_pct_5d,name,weight,latest_close_x,latest_close_y
0,002617.SZ,17.78778,4.7047,露笑科技,1.2121,9.23,9.23
1,002893.SZ,16.45894,3.6503,京能热力,0.1718,12.54,12.54
2,300040.SZ,10.19566,4.6832,九洲集团,0.2602,7.60,7.60
3,000993.SZ,8.40642,3.4771,闽东电力,0.2764,14.88,14.88
4,000862.SZ,7.25514,4.0299,银星能源,0.2844,7.15,7.15
5,600821.SH,7.19872,3.4440,金开新能,0.8758,8.11,8.11
6,601778.SH,6.94392,4.2654,晶科科技,1.1741,5.93,5.93
7,605162.SH,6.39126,3.1895,新中港,0.0954,9.80,9.80
8,600930.SH,6.09810,4.0650,华电新能,1.7739,6.43,6.43
9,600509.SH,6.09552,1.8672,天富能源,0.6344,9.74,9.74


In [10]:
# 换手率均值 >5% 且 过去5天最大涨跌幅 <5%（高换手但未大涨，潜在吸筹）
high_tr   = set(tr_mean[tr_mean['tr_mean_5d'] > 5]['ts_code'])
low_move  = set(max_pct[max_pct['max_pct_5d'] < 5]['ts_code'])

df_hidden = (tr_mean[tr_mean['ts_code'].isin(high_tr & low_move)]
             .merge(max_pct[['ts_code','max_pct_5d']], on='ts_code')
             .merge(df_meta, on='ts_code', how='left')
             .merge(latest_close, on='ts_code', how='left')
             .sort_values('tr_mean_5d', ascending=False)
             .reset_index(drop=True))

print(f"高换手低波动：{len(df_hidden)} 只")
df_hidden

高换手低波动：16 只


,ts_code,tr_mean_5d,max_pct_5d,name,weight,latest_close_x,latest_close_y
0,002617.SZ,17.78778,4.7047,露笑科技,1.2121,9.23,9.23
1,002893.SZ,16.45894,3.6503,京能热力,0.1718,12.54,12.54
2,300040.SZ,10.19566,4.6832,九洲集团,0.2602,7.60,7.60
3,000993.SZ,8.40642,3.4771,闽东电力,0.2764,14.88,14.88
4,000862.SZ,7.25514,4.0299,银星能源,0.2844,7.15,7.15
5,600821.SH,7.19872,3.4440,金开新能,0.8758,8.11,8.11
6,601778.SH,6.94392,4.2654,晶科科技,1.1741,5.93,5.93
7,605162.SH,6.39126,3.1895,新中港,0.0954,9.80,9.80
8,600930.SH,6.09810,4.0650,华电新能,1.7739,6.43,6.43
9,600509.SH,6.09552,1.8672,天富能源,0.6344,9.74,9.74


In [12]:
# ── 画过去3个月 K线+BB+MA+MACD（引用上面已筛出的 df_turnover, df_p）───────
import os, numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ── 指标计算 ──────────────────────────────────────────────────────────────
def calc(df):
    c = df['close']
    df['ma5']      = c.rolling(5).mean()
    df['ma10']     = c.rolling(10).mean()
    df['ma20']     = c.rolling(20).mean()
    df['bb_mid']   = c.rolling(20).mean()
    df['bb_upper'] = df['bb_mid'] + 2 * c.rolling(20).std()
    df['bb_lower'] = df['bb_mid'] - 2 * c.rolling(20).std()
    ema12          = c.ewm(span=12, adjust=False).mean()
    ema26          = c.ewm(span=26, adjust=False).mean()
    df['dif']      = ema12 - ema26
    df['dea']      = df['dif'].ewm(span=9, adjust=False).mean()
    df['bar']      = (df['dif'] - df['dea']) * 2
    return df

# ── 画图 ──────────────────────────────────────────────────────────────────
def draw(ax1, ax2, df, title):
    x = np.arange(len(df))
    for i, (_, r) in enumerate(df.iterrows()):
        c = '#d63031' if r['close'] >= r['open'] else '#00b894'
        ax1.plot([i, i], [r['low'], r['high']], color=c, lw=0.6)
        bh = max(abs(r['close'] - r['open']), r['close'] * 0.001)
        ax1.bar(i, bh, bottom=min(r['open'], r['close']), color=c, width=0.7, lw=0)
    ax1.plot(x, df['ma5'],      color='#0984e3', lw=0.9, label='MA5')
    ax1.plot(x, df['ma10'],     color='#fdcb6e', lw=0.9, label='MA10')
    ax1.plot(x, df['ma20'],     color='#a29bfe', lw=0.9, label='MA20')
    ax1.plot(x, df['bb_upper'], color='#55efc4', lw=0.8, ls='--', label='BB上')
    ax1.plot(x, df['bb_mid'],   color='#55efc4', lw=0.7, ls=':')
    ax1.plot(x, df['bb_lower'], color='#55efc4', lw=0.8, ls='--', label='BB下')
    ax1.fill_between(x, df['bb_upper'], df['bb_lower'], color='#55efc4', alpha=0.07)
    step = max(1, len(df) // 10)
    ax1.set_xticks(x[::step])
    ax1.set_xticklabels([df.index[i].strftime('%m-%d') for i in x[::step]], rotation=30, fontsize=7)
    ax1.legend(fontsize=6, loc='upper left', ncol=4)
    ax1.set_xlim(-1, len(df))
    ax1.set_title(title, fontsize=10, fontweight='bold')
    ax1.grid(True, alpha=0.2, ls='--')

    colors = np.where(df['bar'] >= 0, '#d63031', '#00b894')
    ax2.bar(x, df['bar'], color=colors, width=0.7, lw=0, label='MACD柱')
    ax2.plot(x, df['dif'], color='#0984e3', lw=0.9, label='DIF')
    ax2.plot(x, df['dea'], color='#fdcb6e', lw=0.9, label='DEA')
    ax2.axhline(0, color='gray', lw=0.5, ls='--')
    ax2.set_xticks(x[::step])
    ax2.set_xticklabels([df.index[i].strftime('%m-%d') for i in x[::step]], rotation=30, fontsize=7)
    ax2.legend(fontsize=6, loc='upper left', ncol=3)
    ax2.set_xlim(-1, len(df))
    ax2.grid(True, alpha=0.2, ls='--')



PLOT_OUT = os.path.join(SCRIPT_DIR, 'Astocks', 'Elec', 'plots_hidden')
os.makedirs(PLOT_OUT, exist_ok=True)

plt.rcParams.update({'font.family': 'Microsoft YaHei', 'axes.unicode_minus': False})

# 目标：df_turnover ∪ df_p 的所有股票
target_codes = set(df_hidden['ts_code'])  # set(df_turnover['ts_code']) | set(df_p['ts_code'])
df_meta      = pd.read_csv(STOCKS_CSV, dtype=str)[['ts_code', 'name']]
name_map     = df_meta.set_index('ts_code')['name'].to_dict()
print(f"待画股票：{len(target_codes)} 只")

# ── 主循环 ────────────────────────────────────────────────────────────────
for fname in sorted(os.listdir(DATA_DIR)):
    if not fname.endswith('.csv'):
        continue
    code = fname.split('_')[0]    # 前6位代码
    # 找匹配的 ts_code
    ts = next((t for t in target_codes if t.startswith(code)), None)
    if ts is None:
        continue

    df = pd.read_csv(os.path.join(DATA_DIR, fname))
    df['trade_date'] = pd.to_datetime(df['trade_date'].astype(str))
    df = df.sort_values('trade_date').tail(65).reset_index(drop=True)  # 约3个月
    df.set_index('trade_date', inplace=True)
    for col in ['open','high','low','close']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df.dropna(subset=['open','high','low','close'], inplace=True)
    if len(df) < 20:
        continue
    df = calc(df)

    name  = name_map.get(ts, ts)
    title = f"{name}（{ts}）  日K线 过去3个月  {df.index[0].date()} ~ {df.index[-1].date()}"
    fig   = plt.figure(figsize=(18, 9))
    gs    = GridSpec(2, 1, figure=fig, height_ratios=[3, 1], hspace=0.4)
    draw(fig.add_subplot(gs[0]), fig.add_subplot(gs[1]), df, title)
    fig.savefig(os.path.join(PLOT_OUT, f"{ts}_{name}.png"), bbox_inches='tight', dpi=150)
    plt.close(fig)
    print(f"  {ts} {name}  OK")

print(f"\n完成，图片保存至: {PLOT_OUT}")


待画股票：16 只
  000591.SZ 太阳能  OK
  000722.SZ 湖南发展  OK
  000862.SZ 银星能源  OK
  000993.SZ 闽东电力  OK
  001258.SZ 立新能源  OK
  002617.SZ 露笑科技  OK
  002893.SZ 京能热力  OK
  300040.SZ 九洲集团  OK
  600101.SH 明星电力  OK
  600149.SH 廊坊发展  OK
  600452.SH 涪陵电力  OK
  600509.SH 天富能源  OK
  600821.SH 金开新能  OK
  600930.SH 华电新能  OK
  601778.SH 晶科科技  OK
  605162.SH 新中港  OK

完成，图片保存至: C:\Users\junyu\OneDrive\桌面\Quant\Astock\Astocks\Elec\plots_hidden
